In [ ]:
!pip install -U transformers datasets rouge-score accelerate tensorboard

import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
from rouge_score import rouge_scorer

In [ ]:
# ======================
# Config
# ======================

MODEL_CHECKPOINT = "google-t5/t5-small"   # or t5-base, t5-large
MAX_INPUT_LENGTH = 2048
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = "./t5-summarization-model"
CSV_PATH = ""

prefix = "summarize: "   # or "slangify: " etc.

# ======================
# Load dataset from single CSV and split
# ======================

data_files = {"all": CSV_PATH}
dataset = load_dataset("csv", data_files=data_files)

# 90% train, 10% validation
dataset = dataset["all"].train_test_split(test_size=0.1, seed=42)
dataset["validation"] = dataset["test"]
del dataset["test"]

# ======================
# Tokenizer & model
# ======================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ======================
# Preprocessing
# ======================

def preprocess_function(examples):
    """
    Uses:
      - input:  examples["prompt"]
      - target: examples["GenZ_completion"]
    """
    inputs = [prefix + doc for doc in examples["prompt"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["GenZ_completion"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=dataset["train"].column_names
)

In [ ]:
# ======================
# Metrics (ROUGE)
# ======================

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions
    pred_ids = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 in labels as well
    labels_ids = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    rouge1_list, rouge2_list, rougeL_list = [], [], []
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = rouge.score(label, pred)
        rouge1_list.append(scores["rouge1"].fmeasure)
        rouge2_list.append(scores["rouge2"].fmeasure)
        rougeL_list.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": np.mean(rouge1_list),
        "rouge2": np.mean(rouge2_list),
        "rougeL": np.mean(rougeL_list),
    }

In [ ]:
# ======================
# Training setup
# ======================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    save_steps=500,
    eval_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    fp16=True,
    predict_with_generate=True,
    report_to="tensorboard",
    seed=42
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    return_tensors="pt"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

In [ ]:
# ======================
# Train
# ======================

history = trainer.train()

# Save final model & tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete! Model saved to {OUTPUT_DIR}")

In [ ]:
# ======================
# Inference example
# ======================

fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

fine_tuned_model.to(device)

# Example prompt (same style as your 'prompt' column)
input_text = "Your input text here."

inputs = fine_tuned_tokenizer(
    prefix + input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

summary_ids = fine_tuned_model.generate(
    inputs["input_ids"],
    max_length=128,
    min_length=20,
    num_beams=4,
    early_stopping=True,
    no_repeat_ngram_size=2
)

output_text = fine_tuned_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)
print(f"Model output: {output_text}")